# U-Net on your new dataset

Select the project `.venv` kernel and run all cells. Images are in `data/936/data`; same-name masks are in `data/936/mask`. Defaults: images 1Ã¢â‚¬â€œ5, up to 120 epochs, RGB input, three-class softmax, 128x128 resolution, 16 base filters. Class IDs are preserved as 0, 1, and 2; semantic names are not assumed.

`LIMIT=5` uses the same images for training and evaluation, so metrics are a sanity check. Set `LIMIT=None` for the full dataset with a separate held-out test split.

The multiclass head uses all decoder features. For this tiny sanity run, dropout is disabled, the initial learning rate is 0.001, and training runs for 120 epochs. Outputs are model predictions, not altered ground-truth masks.


In [ ]:
import json, os, sys
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
from PIL import Image
import tensorflow as tf
import matplotlib.pyplot as plt
from model import unet

# Set LIMIT=None for all 936 pairs with a held-out test split.
DATASET = Path('data/936')
LIMIT = 5
EPOCHS = 120
IMAGE_SIZE = 128
BATCH_SIZE = 4
BASE_FILTERS = 16
LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.0 if LIMIT is not None else 0.5
OUTPUT = Path('runs') / ('notebook_five_images_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
args = SimpleNamespace(dataset=DATASET, limit=LIMIT, epochs=EPOCHS,
    size=IMAGE_SIZE, batch_size=BATCH_SIZE, base_filters=BASE_FILTERS, learning_rate=LEARNING_RATE, dropout_rate=DROPOUT_RATE, output=OUTPUT)
if IMAGE_SIZE < 16 or IMAGE_SIZE % 16:
    raise ValueError('IMAGE_SIZE must be a positive multiple of 16')
if min(EPOCHS, BATCH_SIZE, BASE_FILTERS) < 1 or (LIMIT is not None and LIMIT < 1):
    raise ValueError('All parameters must be positive')
print('Python:', sys.executable)
print('TensorFlow:', tf.__version__)
print('Dataset:', DATASET.resolve())


## Inspect and validate image/mask pairs


In [ ]:
args.output.mkdir(parents=True, exist_ok=False)
tf.keras.utils.set_random_seed(42)
images = {x.stem: x for x in (args.dataset / 'data').glob('*.png')}
masks = {x.stem: x for x in (args.dataset / 'mask').glob('*.png')}
if not images or images.keys() != masks.keys():
    raise ValueError('Dataset must contain exactly matched data/*.png and mask/*.png')
keys = sorted(images)
if args.limit is not None:
    keys = sorted(keys, key=lambda k: (0, int(k)) if k.isdigit() else (1, k))
if args.limit is not None:
    if args.limit > len(keys):
        raise ValueError('Requested limit exceeds available image/mask pairs')
    keys = keys[:args.limit]
labels = set()
for key in keys:
    with Image.open(images[key]) as im, Image.open(masks[key]) as mask:
        values = np.asarray(mask)
        if values.ndim != 2 or im.size != mask.size:
            raise ValueError(f'Invalid mask or mismatched dimensions: {key}')
        labels.update(np.unique(values).tolist())
if labels != {0, 1, 2}:
    raise ValueError(f'Expected mask IDs 0,1,2, found {labels}')
rng = np.random.default_rng(42)
order = rng.permutation(len(keys))
ntest = int(np.ceil(len(keys) * .2))
nval = int(np.ceil((len(keys) - ntest) * .1))
splits = {'test': order[:ntest], 'validation': order[ntest:ntest+nval], 'train': order[ntest+nval:]}
if args.limit is not None:
    splits = {name: np.arange(len(keys)) for name in ('train', 'validation', 'test')}
manifest = {name: [keys[i] for i in indices] for name, indices in splits.items()}
(args.output / 'split.json').write_text(json.dumps(manifest, indent=2))
settings = {k: str(v) if isinstance(v, Path) else v for k,v in vars(args).items()}
settings.update(python_executable=sys.executable, evaluation_scope='same training samples - sanity check only' if args.limit else 'held-out test', seed=42, label_ids=[0,1,2], tensorflow=tf.__version__, devices=[str(x) for x in tf.config.list_physical_devices()])
(args.output / 'config.json').write_text(json.dumps(settings, indent=2))



## Load and preprocess images and indexed masks


In [ ]:
def load(image, mask):
    image = tf.io.decode_png(tf.io.read_file(image), channels=3)
    image = tf.image.resize(tf.cast(image, tf.float32), [args.size, args.size]) / 255.
    mask = tf.io.decode_png(tf.io.read_file(mask), channels=1)
    mask = tf.image.resize(mask, [args.size, args.size], method='nearest')
    return image, tf.cast(tf.squeeze(mask, -1), tf.int32)

def dataset(name):
    ids = manifest[name]
    ds = tf.data.Dataset.from_tensor_slices(([str(images[k]) for k in ids], [str(masks[k]) for k in ids]))
    if name == 'train':
        ds = ds.shuffle(len(ids), seed=42)
    return ds.map(load, num_parallel_calls=2).batch(args.batch_size).prefetch(1)



## Train U-Net


In [ ]:
train, val, test = dataset('train'), dataset('validation'), dataset('test')
model = unet(input_size=(args.size,args.size,3), num_classes=3, base_filters=args.base_filters,
             learning_rate=args.learning_rate, dropout_rate=args.dropout_rate)
print('Dataset split:', {k: len(v) for k,v in manifest.items()}, flush=True)
print('Parameters:', model.count_params(), 'Hardware:', settings['devices'], flush=True)
history = model.fit(train, validation_data=val, epochs=args.epochs, verbose=2, shuffle=False, callbacks=[
    tf.keras.callbacks.ModelCheckpoint(str(args.output / 'best.keras'), monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=1e-5),
    tf.keras.callbacks.CSVLogger(str(args.output / 'training.csv'))])
(args.output / 'history.json').write_text(json.dumps(history.history, indent=2))


## Evaluate and save model predictions


In [ ]:
model = tf.keras.models.load_model(args.output / 'best.keras')
metrics = model.evaluate(test, return_dict=True, verbose=2)
confusion = np.zeros((3,3), dtype=np.int64)
pred_dir = args.output / 'predictions'
pred_dir.mkdir()
saved = 0
for x,y in test:
    pred = np.argmax(model(x, training=False).numpy(), axis=-1)
    actual = y.numpy()
    confusion += np.bincount((actual * 3 + pred).ravel(), minlength=9).reshape(3,3)
    for j in range(len(pred)):
        if saved < 6:
            key = manifest['test'][saved]
            Image.fromarray(pred[j].astype('uint8')).save(pred_dir / f'{key}_mask.png')
            palette = np.array([[0,0,0],[255,100,0],[0,200,80]], dtype='uint8')
            Image.fromarray(palette[pred[j]]).save(pred_dir / f'{key}_color.png')
            saved += 1
denom = confusion.sum(0) + confusion.sum(1)
dice = np.divide(2*np.diag(confusion), denom, out=np.zeros(3), where=denom!=0)
print("Predicted pixel counts by class ID:", confusion.sum(axis=0).tolist())
metrics.update(predicted_pixels_by_class_id=confusion.sum(axis=0).tolist(), evaluation_scope=settings['evaluation_scope'], dice_by_class_id=dice.tolist(), macro_dice=float(dice.mean()), confusion_matrix=confusion.tolist())
(args.output / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print('Evaluation metrics:', json.dumps(metrics), flush=True)
print('Artifacts:', args.output.resolve(), flush=True)


## Compare images, ground truth, and predictions
This five-image run is intentionally trained longer to check whether the model can fit the samples; same-sample metrics do not measure unseen-image performance.


In [ ]:
ids = manifest['test'][:5]
fig, axes = plt.subplots(len(ids), 3, figsize=(12, 3 * len(ids)), squeeze=False)
for row, key in enumerate(ids):
    with Image.open(images[key]) as image:
        axes[row, 0].imshow(image.convert('RGB'))
    with Image.open(masks[key]) as mask:
        axes[row, 1].imshow(np.asarray(mask), cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
    with Image.open(pred_dir / f'{key}_mask.png') as prediction:
        axes[row, 2].imshow(np.asarray(prediction), cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
    for col, title in enumerate((f'Image {key}', 'Ground truth', 'Prediction')):
        axes[row, col].set_title(title)
        axes[row, col].axis('off')
plt.tight_layout()
fig.savefig(args.output / 'prediction_comparison.png', dpi=140)
plt.show()
print('Artifacts:', args.output.resolve())
